In [ ]:
from google.colab import files
import zipfile
import os

# Upload the dataset zip
uploaded = files.upload()

# Get the uploaded file name
zip_filename = list(uploaded.keys())[0]

# Unzip
with zipfile.ZipFile(zip_filename, 'r') as zip_ref:
    zip_ref.extractall("/content/dataset")

# Check files
!ls /content/dataset

Saving obstacle detection zip.zip to obstacle detection zip.zip
test  train  valid


In [ ]:
!git clone https://github.com/ultralytics/yolov5.git
%cd yolov5
!pip install -r requirements.txt

Cloning into 'yolov5'...
remote: Enumerating objects: 17524, done.
remote: Counting objects: 100% (26/26), done.
remote: Compressing objects: 100% (26/26), done.
remote: Total 17524 (delta 11), reused 0 (delta 0), pack-reused 17498 (from 4)
Receiving objects: 100% (17524/17524), 16.61 MiB | 11.03 MiB/s, done.
Resolving deltas: 100% (12000/12000), done.
/content/yolov5
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 363.4/363.4 MB 4.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 13.8/13.8 MB 112.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 24.6/24.6 MB 92.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 883.7/883.7 kB 59.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 664.8/664.8 MB 2.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 211.5/211.5 MB 5.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.3/56.3 MB 17.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 127.9/127.9 MB 7.5 MB/s et

In [ ]:
import glob
import os

# Path to your dataset
dataset_dir = "/content/dataset"

# 1️⃣ Find all class IDs in labels
label_files = glob.glob(os.path.join(dataset_dir, "train", "labels", "*.txt"))
all_labels = set()

for file in label_files:
    with open(file) as f:
        for line in f:
            cls = int(line.split()[0])
            all_labels.add(cls)

# 2️⃣ Sort class IDs and count
classes_sorted = sorted(all_labels)
num_classes = len(classes_sorted)

print(f"Classes found: {classes_sorted}")
print(f"Number of classes: {num_classes}")

# 3️⃣ Create placeholder names
names_list = [f"class{i}" for i in range(num_classes)]

# 4️⃣ Create data.yaml content
yaml_content = f"""
train: {dataset_dir}/train/images
val: {dataset_dir}/valid/images
test: {dataset_dir}/test/images

nc: {num_classes}
names: {names_list}
"""

# 5️⃣ Save data.yaml
yaml_path = "/content/yolov5/data.yaml"
with open(yaml_path, "w") as f:
    f.write(yaml_content)

print(f"✅ data.yaml created at {yaml_path}")


Classes found: [0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 17, 18, 19, 20, 21]
Number of classes: 22
✅ data.yaml created at /content/yolov5/data.yaml


In [ ]:
!WANDB_MODE=disabled python train.py --img 640 --batch 16 --epochs 50 --data /content/yolov5/data.yaml --weights yolov5s.pt


Streaming output truncated to the last 5000 lines.
  with torch.cuda.amp.autocast(amp):
      33/49      4.58G    0.02419    0.01611   0.006065         37        640:  39% 58/147 [00:24<00:34,  2.60it/s]/content/yolov5/train.py:414: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast(amp):
      33/49      4.58G    0.02413     0.0161   0.006052         27        640:  40% 59/147 [00:25<00:36,  2.44it/s]/content/yolov5/train.py:414: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast(amp):
      33/49      4.58G    0.02411    0.01613   0.006019         37        640:  41% 60/147 [00:25<00:32,  2.70it/s]/content/yolov5/train.py:414: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast(amp

In [ ]:
import glob

# Find the latest best.pt
weight_files = glob.glob("runs/train/*/weights/best.pt")
latest_best = max(weight_files, key=os.path.getctime)

print(f"Using weights: {latest_best}")

# Run evaluation on the test set
!python val.py --weights "{latest_best}" --data data.yaml --task test


Using weights: runs/train/exp/weights/best.pt
val: data=data.yaml, weights=['runs/train/exp/weights/best.pt'], batch_size=32, imgsz=640, conf_thres=0.001, iou_thres=0.6, max_det=300, task=test, device=, workers=8, single_cls=False, augment=False, verbose=False, save_txt=False, save_hybrid=False, save_conf=False, save_json=False, project=runs/val, name=exp, exist_ok=False, half=False, dnn=False
YOLOv5 🚀 v7.0-423-g567c6646 Python-3.11.13 torch-2.6.0+cu124 CUDA:0 (Tesla T4, 15095MiB)

Fusing layers... 
Model summary: 157 layers, 7069459 parameters, 0 gradients, 15.9 GFLOPs
test: Scanning /content/dataset/test/labels... 504 images, 0 backgrounds, 0 corrupt: 100% 504/504 [00:01<00:00, 479.60it/s]
test: New cache created: /content/dataset/test/labels.cache
                 Class     Images  Instances          P          R      mAP50   mAP50-95: 100% 16/16 [00:09<00:00,  1.62it/s]
                   all        504        504      0.714      0.814      0.817      0.569
                class0  

In [ ]:
!python val.py --weights "{latest_best}" --data data.yaml --task test --save-txt --save-conf --save-hybrid


val: data=data.yaml, weights=['runs/train/exp/weights/best.pt'], batch_size=32, imgsz=640, conf_thres=0.001, iou_thres=0.6, max_det=300, task=test, device=, workers=8, single_cls=False, augment=False, verbose=False, save_txt=True, save_hybrid=True, save_conf=True, save_json=False, project=runs/val, name=exp, exist_ok=False, half=False, dnn=False
WARNING ⚠️ --save-hybrid will return high mAP from hybrid labels, not from predictions alone
YOLOv5 🚀 v7.0-423-g567c6646 Python-3.11.13 torch-2.6.0+cu124 CUDA:0 (Tesla T4, 15095MiB)

Fusing layers... 
Model summary: 157 layers, 7069459 parameters, 0 gradients, 15.9 GFLOPs
test: Scanning /content/dataset/test/labels.cache... 504 images, 0 backgrounds, 0 corrupt: 100% 504/504 [00:00<?, ?it/s]
                 Class     Images  Instances          P          R      mAP50   mAP50-95: 100% 16/16 [00:12<00:00,  1.30it/s]
                   all        504        504          1          1      0.995      0.995
                class0        504         1

In [ ]:
# ---------- Run this in your Colab where /content/yolov5 exists ----------
import os, glob, subprocess, yaml
from PIL import Image
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# ----- Config (edit only if needed) -----
yolov5_root = "/content/yolov5"                       # should point to your yolov5 clone
weights_hint = "runs/train/exp/weights/best.pt"       # your best.pt
data_yaml_hint = "data.yaml"                         # data.yaml location (relative to yolov5_root)
project = "runs/val"
name = "conf_eval"
iou_threshold = 0.5
# -----------------------------------------

# sanity
if not os.path.isdir(yolov5_root):
    raise FileNotFoundError(f"Could not find yolov5 at {yolov5_root}. Run in the Colab where you cloned yolov5.")

os.chdir(yolov5_root)

# locate weights
weights_path = os.path.join(yolov5_root, weights_hint)
if not os.path.exists(weights_path):
    found = glob.glob(os.path.join(yolov5_root, "runs", "train", "*", "weights", "best.pt"))
    if found:
        weights_path = found[-1]
        print("Found weights at:", weights_path)
    else:
        raise FileNotFoundError("Could not find best.pt. Please set weights_hint correctly.")

# locate data.yaml
data_yaml = os.path.join(yolov5_root, data_yaml_hint)
if not os.path.exists(data_yaml):
    found = glob.glob(os.path.join(yolov5_root, "**", "data.yaml"), recursive=True)
    if found:
        data_yaml = found[0]
        print("Found data.yaml at:", data_yaml)
    elif os.path.exists("data.yaml"):
        data_yaml = os.path.abspath("data.yaml")
        print("Using data.yaml from CWD:", data_yaml)
    else:
        raise FileNotFoundError("Could not find data.yaml. Ensure it's present and points to your test images.")

print("yolov5 root:", yolov5_root)
print("weights:", weights_path)
print("data.yaml:", data_yaml)

# Try running val.py with the correct flag (--exist-ok). If that fails, run without it.
cmd_try = [
    "python", "val.py",
    "--weights", weights_path,
    "--data", data_yaml,
    "--task", "test",
    "--save-txt",
    "--save-conf",
    "--project", project,
    "--name", name,
    "--exist-ok"
]

print("Running (first attempt) with --exist-ok...")
proc = subprocess.run(cmd_try, capture_output=True, text=True)
if proc.returncode != 0:
    print("First attempt failed. stderr (truncated):\n", proc.stderr[-1000:])
    # Retry without --exist-ok if first attempt failed due to arg parsing
    cmd_fallback = [
        "python", "val.py",
        "--weights", weights_path,
        "--data", data_yaml,
        "--task", "test",
        "--save-txt",
        "--save-conf",
        "--project", project,
        "--name", name
    ]
    print("Retrying without --exist-ok...")
    proc2 = subprocess.run(cmd_fallback, capture_output=True, text=True)
    print("Second attempt stdout (last 200 lines):\n", "\n".join(proc2.stdout.splitlines()[-200:]))
    if proc2.returncode != 0:
        print("Second attempt also failed. stderr (last 1000 chars):\n", proc2.stderr[-1000:])
        raise RuntimeError("val.py failed in both attempts. Inspect the printed stderr above.")
    else:
        print("val.py finished (fallback).")
else:
    print("val.py finished (first attempt).")
    print("\nLast lines of stdout:\n", "\n".join(proc.stdout.splitlines()[-200:]))

# locate the predictions folder created under runs/val
val_runs = sorted(glob.glob(os.path.join(yolov5_root, project, "*")), key=os.path.getctime)
if not val_runs:
    raise FileNotFoundError("No runs found in runs/val. Did val.py run successfully?")
pred_folder = val_runs[-1]
pred_labels_folder = os.path.join(pred_folder, "labels")
print("Predicted labels folder:", pred_labels_folder)

# parse data.yaml to find test image folder and class names
with open(data_yaml) as f:
    data = yaml.safe_load(f)
test_images_path = data.get("test")
if test_images_path is None:
    raise ValueError("data.yaml missing 'test:' entry.")
if not os.path.isabs(test_images_path):
    test_images_path = os.path.join(yolov5_root, test_images_path)
# derive labels folder from images folder
if "images" in test_images_path:
    test_labels_folder = test_images_path.replace("images", "labels")
else:
    test_labels_folder = os.path.join(os.path.dirname(test_images_path), "labels")
print("Test labels folder:", test_labels_folder)

names = data.get("names")
if isinstance(names, dict):
    names = [names[str(i)] if str(i) in names else names[i] for i in range(len(names))]
if names is None:
    nc = data.get("nc")
    if nc is None:
        raise ValueError("Could not determine classes from data.yaml")
    names = [f"class{i}" for i in range(nc)]
nc = len(names)
print(f"Detected {nc} classes.")

# helper conversion / IoU
def yolo_to_xyxy(line, img_w, img_h):
    parts = [float(x) for x in line.split()]
    cls = int(parts[0])
    x_c, y_c, w, h = parts[1:5]
    x1 = (x_c - w/2) * img_w
    y1 = (y_c - h/2) * img_h
    x2 = (x_c + w/2) * img_w
    y2 = (y_c + h/2) * img_h
    return cls, [max(0,x1), max(0,y1), min(img_w,x2), min(img_h,y2)]

def iou(boxA, boxB):
    xA = max(boxA[0], boxB[0])
    yA = max(boxA[1], boxB[1])
    xB = min(boxA[2], boxB[2])
    yB = min(boxA[3], boxB[3])
    interW = max(0, xB - xA)
    interH = max(0, yB - yA)
    interArea = interW * interH
    areaA = max(0,(boxA[2]-boxA[0])) * max(0,(boxA[3]-boxA[1]))
    areaB = max(0,(boxB[2]-boxB[0])) * max(0,(boxB[3]-boxB[1]))
    union = areaA + areaB - interArea
    return interArea/union if union>0 else 0.0

# build confusion matrix (nc+1 x nc+1), last index = background
conf_mat = np.zeros((nc+1, nc+1), dtype=int)

# collect all test images
test_images_folder = test_images_path if os.path.isdir(test_images_path) else os.path.dirname(test_images_path)
image_files = sorted(glob.glob(os.path.join(test_images_folder, "*.*")))
print(f"Found {len(image_files)} test images.")

for img_path in image_files:
    base = os.path.splitext(os.path.basename(img_path))[0]
    gt_file = os.path.join(test_labels_folder, base + ".txt")
    pred_file = os.path.join(pred_labels_folder, base + ".txt")
    try:
        w,h = Image.open(img_path).size
    except Exception:
        continue
    gt_boxes=[]
    if os.path.exists(gt_file):
        with open(gt_file) as f:
            for line in f.read().strip().splitlines():
                if not line.strip(): continue
                cls, bbox = yolo_to_xyxy(line, w, h)
                gt_boxes.append({"cls":cls, "bbox":bbox, "matched":False})
    pred_boxes=[]
    if os.path.exists(pred_file):
        with open(pred_file) as f:
            for line in f.read().strip().splitlines():
                if not line.strip(): continue
                parts=line.split()
                cls = int(float(parts[0]))
                conf = float(parts[5]) if len(parts)>=6 else 1.0
                _, bbox = yolo_to_xyxy(line, w, h)
                pred_boxes.append({"cls":cls, "bbox":bbox, "conf":conf, "matched":False})
    # greedy match by IoU
    pairs=[]
    for pi,p in enumerate(pred_boxes):
        for gi,g in enumerate(gt_boxes):
            val=iou(p["bbox"], g["bbox"])
            if val >= iou_threshold:
                pairs.append((val, pi, gi))
    pairs.sort(reverse=True, key=lambda x:x[0])
    matched_p=set(); matched_g=set()
    for val,pi,gi in pairs:
        if pi in matched_p or gi in matched_g: continue
        matched_p.add(pi); matched_g.add(gi)
        pred_cls=pred_boxes[pi]["cls"]; gt_cls=gt_boxes[gi]["cls"]
        conf_mat[gt_cls, pred_cls] += 1
        pred_boxes[pi]["matched"]=True; gt_boxes[gi]["matched"]=True
    # unmatched preds -> FP (true background)
    for p in pred_boxes:
        if not p["matched"]:
            conf_mat[nc, p["cls"]] += 1
    # unmatched gts -> FN (predicted background)
    for g in gt_boxes:
        if not g["matched"]:
            conf_mat[g["cls"], nc] += 1

# plot
labels = names + ["background"]
plt.figure(figsize=(12,10))
sns.heatmap(conf_mat, annot=True, fmt="d", xticklabels=labels, yticklabels=labels, cmap="Blues")
plt.xlabel("Predicted"); plt.ylabel("Actual"); plt.title(f"Detection Confusion Matrix (IoU≥{iou_threshold})")
plt.tight_layout()
plt.show()

total_TP = conf_mat.trace()
total_FP = conf_mat[:-1,:].sum() - total_TP
total_FN = conf_mat[:,:-1].sum() - total_TP
print(f"Summary: TP={total_TP}, FP={total_FP}, FN={total_FN}")
print("Matrix in variable 'conf_mat'; class names in 'names'.")


yolov5 root: /content/yolov5
weights: /content/yolov5/runs/train/exp/weights/best.pt
data.yaml: /content/yolov5/data.yaml
Running (first attempt) with --exist-ok...
val.py finished (first attempt).

Last lines of stdout:
 
Predicted labels folder: /content/yolov5/runs/val/conf_eval/labels
Test labels folder: /content/dataset/test/labels
Detected 22 classes.
Found 504 test images.
Summary: TP=138, FP=366, FN=12237
Matrix in variable 'conf_mat'; class names in 'names'.


In [ ]:
from google.colab import files
files.download('/content/yolov5/runs/train/exp/weights/best.pt')


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>